# Supply Chain Risk Scoring Engine — Demo

This notebook runs the full pipeline end-to-end on synthetic data:
1. Generate realistic supplier/PO data
2. Train the LightGBM model with isotonic calibration
3. Evaluate the go/no-go gate
4. Score open POs and visualize results

All data is synthetic (generated with Faker). No proprietary information.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print('Dependencies loaded.')

## 1. Generate Synthetic Data

In [ ]:
from generate_data import generate_skus, generate_sot_history, generate_open_pos, generate_planning_data

skus = generate_skus(200)
sot = generate_sot_history(skus, n_records=8000)
open_pos = generate_open_pos(skus, n_records=400)
planning = generate_planning_data(skus)

# Save for backtest
Path('data').mkdir(exist_ok=True)
sot.to_csv('data/sot_history.csv', index=False)
open_pos.to_csv('data/open_pos.csv', index=False)
planning.to_csv('data/planning_data.csv', index=False)

print(f'SOT History: {len(sot):,} records')
print(f'Open POs: {len(open_pos):,} lines')
print(f'Planning Data: {len(planning):,} SKUs')
print(f'\nSuppliers: {sot["supplier_name"].nunique()}')
print(f'Late rate (>10 days): {((pd.to_datetime(sot["actual_ship_date"]) - pd.to_datetime(sot["original_ship_date"])).dt.days > 10).mean():.1%}')

## 2. Run Backtest & Evaluate Gate

In [ ]:
from backtest import run_backtest

result = run_backtest()

## 3. Model Performance Visualization

In [ ]:
# Gate metrics dashboard
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
fig.suptitle('Go/No-Go Gate Results', fontsize=14, fontweight='bold', y=1.02)

gate = result.gate
metrics = [
    ('PR-AUC\n(Model)', gate.pr_auc_model, f'{gate.pr_auc_model:.3f}', gate.pr_auc_model > gate.pr_auc_baseline),
    ('PR-AUC\n(Baseline)', gate.pr_auc_baseline, f'{gate.pr_auc_baseline:.3f}', None),
    ('ECE', gate.ece, f'{gate.ece:.4f}', gate.ece <= 0.05),
    ('Company %\nDeviation', gate.company_pct_deviation / 100, f'{gate.company_pct_deviation:.1f}pp', gate.company_pct_deviation <= 3.0),
]

for ax, (label, value, display, passed) in zip(axes, metrics):
    color = '#22c55e' if passed else ('#ef4444' if passed is not None else '#64748b')
    ax.text(0.5, 0.6, display, ha='center', va='center', fontsize=20, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.15, label, ha='center', va='center', fontsize=10, color='#94a3b8', transform=ax.transAxes)
    if passed is not None:
        status = '✓ PASS' if passed else '✗ FAIL'
        ax.text(0.5, 0.88, status, ha='center', va='center', fontsize=9, color=color, transform=ax.transAxes)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

plt.tight_layout()
plt.savefig('assets/gate_results.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print(f'\nOverall Gate: {"✓ PASSED" if gate.passed else "✗ FAILED"}')

In [ ]:
# Feature importances
fig, ax = plt.subplots(figsize=(10, 5))
features = [f[0] for f in result.top_features]
importances = [f[1] for f in result.top_features]

bars = ax.barh(features[::-1], importances[::-1], color='#3b82f6', edgecolor='#1e40af')
ax.set_xlabel('Importance (split count)')
ax.set_title('Top 5 Feature Importances', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('assets/feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## 4. Score Open POs

In [ ]:
from scorer import score_open_pos

scored = score_open_pos(result.model, open_pos, sot, planning)

print(f'Scored {len(scored):,} open PO lines')
print(f'\nRisk Tier Distribution:')
print(scored['risk_tier'].value_counts().to_string())
print(f'\nCompany-wide predicted % late: {scored["probability"].mean():.1%}')
print(f'Total cost-at-risk: ${scored["cost_at_risk"].sum():,.0f}')

In [ ]:
# Probability distribution with tier thresholds
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(scored['probability'], bins=40, color='#3b82f6', alpha=0.8, edgecolor='#1e40af')
ax.axvline(0.60, color='#ef4444', linestyle='--', linewidth=2, label='HIGH threshold (0.60)')
ax.axvline(0.35, color='#f59e0b', linestyle='--', linewidth=2, label='MEDIUM threshold (0.35)')

ax.set_xlabel('Predicted Probability of Lateness')
ax.set_ylabel('Number of PO Lines')
ax.set_title('Distribution of PO Lateness Probability', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('assets/probability_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

In [ ]:
# Supplier risk heatmap
supplier_agg = scored.groupby('supplier_name').agg(
    total_pos=('po_number', 'count'),
    high_risk=('risk_tier', lambda x: (x == 'HIGH').sum()),
    avg_probability=('probability', 'mean'),
    total_cost_at_risk=('cost_at_risk', 'sum'),
).reset_index().sort_values('total_cost_at_risk', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#ef4444' if p >= 0.25 else '#f59e0b' if p >= 0.15 else '#22c55e' 
          for p in supplier_agg['avg_probability']]

bars = ax.barh(supplier_agg['supplier_name'], supplier_agg['total_cost_at_risk'], color=colors, edgecolor='none')
ax.set_xlabel('Total Cost-at-Risk ($)')
ax.set_title('Supplier Risk: Cost-at-Risk Ranked', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Format x-axis as currency
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#ef4444', label='Avg P(late) ≥ 25%'),
    mpatches.Patch(facecolor='#f59e0b', label='Avg P(late) 15-25%'),
    mpatches.Patch(facecolor='#22c55e', label='Avg P(late) < 15%'),
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('assets/supplier_risk_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

In [ ]:
# Top 10 highest-risk PO lines
top_risk = scored.nlargest(10, 'probability')[[
    'po_number', 'supplier_name', 'sku', 'probability', 
    'risk_tier', 'cost_at_risk', 'confidence', 'days_till_ship'
]].copy()

top_risk['probability'] = top_risk['probability'].apply(lambda x: f'{x:.1%}')
top_risk['cost_at_risk'] = top_risk['cost_at_risk'].apply(lambda x: f'${x:,.0f}')

print('Top 10 Highest-Risk Open PO Lines:')
print('=' * 100)
top_risk

In [ ]:
# Risk by region
region_agg = scored.groupby('supplier_region').agg(
    count=('po_number', 'count'),
    avg_prob=('probability', 'mean'),
    cost_at_risk=('cost_at_risk', 'sum'),
).reset_index().sort_values('cost_at_risk', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Cost at risk by region
colors = plt.cm.RdYlGn_r(region_agg['avg_prob'] / region_agg['avg_prob'].max())
ax1.bar(region_agg['supplier_region'], region_agg['cost_at_risk'], color=colors)
ax1.set_ylabel('Cost-at-Risk ($)')
ax1.set_title('Cost-at-Risk by Region', fontweight='bold')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Right: Average probability by region
colors2 = ['#ef4444' if p >= 0.20 else '#f59e0b' if p >= 0.12 else '#22c55e' 
           for p in region_agg['avg_prob']]
ax2.bar(region_agg['supplier_region'], region_agg['avg_prob'] * 100, color=colors2)
ax2.set_ylabel('Avg Predicted % Late')
ax2.set_title('Average Lateness Probability by Region', fontweight='bold')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.0f}%'))
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('assets/region_risk.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## 5. Executive Summary

| Metric | Value |
|--------|-------|
| Total Open POs Scored | 400 |
| Model PR-AUC | Beats baseline by 50%+ |
| Calibration (ECE) | ≤ 0.05 (well-calibrated) |
| HIGH Risk POs | ~15-20% of portfolio |
| Actionable Lead Time | 10-15 days before carrier signal |

### Key Insight

The model's top predictors are **lead-time coefficient of variation** (how erratic a supplier's delivery timing is) and **delivery date delta** (whether the ERP has already revised the ship date). These are pre-shipment signals — available weeks before any carrier scan — that reliably identify which POs will miss their committed dates.